In [ ]:
import os
import faiss
import numpy as np
import pprint
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic import hub
from uuid import uuid4

from dotenv import load_dotenv
load_dotenv()

True

## Configurations

In [3]:
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [4]:
openai_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
google_embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [6]:
# openai_embeddings.embed_query("Hello world")

In [8]:
# google_embeddings.embed_query("Hello world")

In [ ]:
model = ChatGoogleGenerativeAI(model='gemini-3.8-flash')
openai_model = ChatOpenAI(model="gpt-4o", temperature=0)

In [14]:
model.invoke("Who won the 2020 US presidential election?")

AIMessage(content=[{'type': 'text', 'text': '**Joe Biden** won the 2020 US presidential election, defeating the incumbent president, Donald Trump.', 'extras': {'signature': 'EpUFCpIFAWkUfRNq22680Q2aed/4OGm/UZBnjHeiJCfYvXPzKm7ub5A+AqfxurSihFnRKKgyjHnhy8rLFOg7S9KuAxkiLiL/ZXacGn0L+tyRWywOjqcMf8bLDAPDtg/d/erbY7OKQ+R9dHvbX5L0dE8Rg0dfgQg3dhsGQ99izHVFhlLexHnzv4WkSYIfohK7ibqE8D0YQCmM1IYRydjSWzED66/P5Nl79jF75V8UR3v8Vk73tKJjY4L2S5DfXzoXZ7jVbTSTtM7M/m9fdgXSUuHNLjT933avm21FHHi8MVpogaAaFVwWjdRcZ8WgzWwfQNAHDMD66KNp6nHry1j85Ze3LIU6Br2pZ6kA6EGa4WcWAOKVduCYWhCAcGzx0kvifsTgaitBax0HuTSLXuY0xT2LZ3eph7rbRp3piccV7M2vfawaRVEbrno30cuK6b+fJa0HWHF5QU0AH2+E3/hbAFQSN4pLO7DiqU10ccsHy6dOfvWAqoqbRcdlmFVOkG1XNmGJqEKxQpHg8DDHmYhmLTkCkkgdx6vRIjW8e6Y36aaCAqqF4qlD1NISysrRio/T2xqhyi20KF/rsUgn+TGIocRYTA9G3p3sWE2tkvvEfitgwqsr+xzZ+fA6z3utXJm7isTDX8QYY6JhDGmgHojX0lCH9pZShkJsxZJmA7hmTn8JrDzrSC72OGzrv2nTnIEXjQmvRnyULz9Ur/UWZ86pplyVd7Wy4/F3Pa8qMfyG16p3PyDywsqD8j+cgYfgrNJnLNBuyeZUkx73Re5/tAOLUrl1SDfq5FvncRew97LAmLatHgT7kc65S0cITX

## Understanding How Similarity Works

In [15]:
documents=[
    "Washington is the capital of USA",
    "Donald Trump is a president of USA",
    "London is the capital of England."
]

In [16]:
query  = "Who is a president of USA?"

In [17]:
embedded_docs = google_embeddings.embed_documents(documents)
embedded_query = google_embeddings.embed_query(query)

In [18]:
cosine_similarity([embedded_query], embedded_docs)

array([[0.67442095, 0.77875809, 0.56658364]])

In [19]:
euclidean_distances([embedded_query], embedded_docs)

array([[0.80694368, 0.66519456, 0.93103854]])

## Setting Vector Store

In [20]:
len(google_embeddings.embed_query("Hello world"))

3072

In [21]:
index = faiss.IndexFlatL2(3072)

In [22]:
vector_store = FAISS(
    embedding_function = google_embeddings,
    index = index,
    docstore = InMemoryDocstore(),
    index_to_docstore_id={}
)

In [23]:
documents=[
    "Washington is the capital of USA",
    "Donald Trump is a president of USA",
    "London is the capital of England."
]

vector_store.add_texts(documents)

['8d94757d-1cb1-4a96-aca6-c3eace9edd31',
 '6f7d8f50-7dca-46f7-827c-3fb142aa9ece',
 '2e6bd63d-fcf5-489f-aa4a-4b4b49ab6522']

In [24]:
vector_store.index_to_docstore_id

{0: '8d94757d-1cb1-4a96-aca6-c3eace9edd31',
 1: '6f7d8f50-7dca-46f7-827c-3fb142aa9ece',
 2: '2e6bd63d-fcf5-489f-aa4a-4b4b49ab6522'}

### Retrieving Document by Index ID

In [25]:
faiss_index_id = 1

docstore_id = vector_store.index_to_docstore_id[faiss_index_id]

In [26]:
vector_store.docstore.search(docstore_id)

Document(id='6f7d8f50-7dca-46f7-827c-3fb142aa9ece', metadata={}, page_content='Donald Trump is a president of USA')

In [27]:
vector_store.docstore.search(docstore_id).page_content

'Donald Trump is a president of USA'

In [ ]:
vector_store.docstore.search(docstore_id).metadata

{}

In [29]:
vector_store.index.reconstruct(faiss_index_id)

array([-0.01705427,  0.01364006,  0.02891811, ...,  0.00777903,
        0.00772134, -0.0054425 ], shape=(3072,), dtype=float32)

In [30]:
vector_store.index.reconstruct(faiss_index_id).shape

(3072,)

### Similarity Search

In [31]:
vector_store.similarity_search("Who is a president of USA?", k=1)

[Document(id='6f7d8f50-7dca-46f7-827c-3fb142aa9ece', metadata={}, page_content='Donald Trump is a president of USA')]

In [32]:
vector_store.similarity_search("Who is a president of USA?", k=2)

[Document(id='6f7d8f50-7dca-46f7-827c-3fb142aa9ece', metadata={}, page_content='Donald Trump is a president of USA'),
 Document(id='8d94757d-1cb1-4a96-aca6-c3eace9edd31', metadata={}, page_content='Washington is the capital of USA')]

## Preparing LangChain Documents

In [33]:
document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

### Adding Documents to Vector Store

In [34]:
vector_store.add_documents(documents=documents)

['5f9abbdf-1806-4e79-8370-1b6785c08767',
 '0ce87a0e-2e47-4e8a-b5ae-c7cc38e91e6c',
 '977bee1a-a633-4f59-a245-a05ef9d4b7ec',
 'c1bd843a-1347-492d-9604-697f35feb6c2',
 '6f674dc4-331c-42b0-a933-961db4b660cc',
 '8cf86486-776f-40bc-999f-ae7472992b86',
 'b13c6f3e-3039-472a-8a84-fd64fbf36545',
 'd24d565c-4c02-46f4-b10d-f6cb8ecd3427',
 'b2062f34-846d-4db7-950a-121e232d9d63',
 '3f603482-8fda-45f6-9896-0e5c2a20e89d']

### Similarity Search

In [35]:
query = "LangChain provides abstractions to make working with LLMs easy"

vector_store.similarity_search(query, k=5)

[Document(id='977bee1a-a633-4f59-a245-a05ef9d4b7ec', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='d24d565c-4c02-46f4-b10d-f6cb8ecd3427', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='5f9abbdf-1806-4e79-8370-1b6785c08767', metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(id='b2062f34-846d-4db7-950a-121e232d9d63', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.'),
 Document(id='2e6bd63d-fcf5-489f-aa4a-4b4b49ab6522', metadata={}, page_content='London is the capital of England.')]

In [36]:
query = "LangChain provides abstractions to make working with LLMs easy"

vector_store.similarity_search(query, k=3, filter={"source": "tweet"})

[Document(id='977bee1a-a633-4f59-a245-a05ef9d4b7ec', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='d24d565c-4c02-46f4-b10d-f6cb8ecd3427', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='5f9abbdf-1806-4e79-8370-1b6785c08767', metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.')]

In [37]:
query = "LangChain provides abstractions to make working with LLMs easy"

vector_store.similarity_search(query, k=3, filter={"source": "news"})

[Document(id='b2062f34-846d-4db7-950a-121e232d9d63', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.'),
 Document(id='0ce87a0e-2e47-4e8a-b5ae-c7cc38e91e6c', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='c1bd843a-1347-492d-9604-697f35feb6c2', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]

## Saving Vector Store Locally

In [38]:
vector_store.save_local("faiss_index")

## Loading Vector Store

In [ ]:
FAISS.load_local("faiss_index", openai_embeddings, allow_dangerous_deserialization=True)

## Creating Index using HNSW

In [ ]:
dimension = 3072
approximate_graph_connections = 32

faiss_index = faiss.IndexHNSWFlat(
    dimension,
    approximate_graph_connections,
    faiss.METRIC_INNER_PRODUCT
)

# Higher value = better recall, but slower indexing
faiss_index.hnsw.efConstruction = 200

# Higher value = better recall, but slower search
faiss_index.hnsw.efSearch = 64

### Loading PDF Document

In [42]:
file_path = "data\\llama2-research-paper.pdf"

In [43]:
loader = PyPDFLoader(file_path)
pages = []
async for page in loader.alazy_load():
    pages.append(page)

Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


In [45]:
# pages

### Performing Chunking

In [46]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

In [47]:
chunks = text_splitter.split_documents(pages)

In [48]:
print("Total pages:", len(pages))
print("Total chunks:", len(chunks))

Total pages: 77
Total chunks: 343


In [49]:
print(chunks[0].page_content)

Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojnic


In [50]:
print(chunks[0].metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}


### Creating Vector Store

In [53]:
vector_store = FAISS(
    embedding_function=openai_embeddings,
    index=faiss_index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
    normalize_L2=False
)

In [61]:
ids = vector_store.add_documents(chunks)

print("Documents added:", len(ids))
print("Total vectors:", vector_store.index.ntotal)

Documents added: 343
Total vectors: 343


### Creating Retriever

In [62]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

In [63]:
query = "What is the architecture of Llama 2?"

retrieved_documents = retriever.invoke(query)

for i, document in enumerate(retrieved_documents, start=1):
    print(f"\n--- Result {i} ---")
    print(document.page_content[:500])
    print("Metadata:", document.metadata)


--- Result 1 ---
1. Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data. We also
increased the size of the pretraining corpus by 40%, doubled the context length of the model, and
adopted grouped-query attention (Ainslie et al., 2023). We are releasing variants ofLlama 2 with
7B, 13B, and 70B parameters. We have also trained 34B variants, which we report on in this paper
but are not releasing.§
2. Llama 2-Chat, a fine-tuned version ofLlama 2 that is optimized for dialogue use ca
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 3, 'page_label': '4'}

--- Result 2 ---
Llama 2.
Hardware and

In [64]:
vector_store.save_local("faiss_index_llama2")

### Pulling Prompt from Hub

In [66]:
prompt = hub.pull("rlm/rag-prompt")

pprint.pprint(prompt.messages)

[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]


In [67]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

### Creating Chain

In [68]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [69]:
rag_chain.invoke("What is Task Decomposition?")

"I don't know. The provided context does not contain information about Task Decomposition."

## Creating Another HNSW Index with L2 Metric

In [70]:
dimension

3072

In [71]:
approximate_graph_connections

32

In [73]:
faiss_index = faiss.IndexHNSWFlat(
    dimension,
    approximate_graph_connections,
    faiss.METRIC_L2
)

faiss_index.hnsw.efConstruction = 200
faiss_index.hnsw.efSearch = 64

In [74]:
vector_store = FAISS(
    embedding_function=openai_embeddings,
    index=faiss_index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
    normalize_L2=False
)

In [75]:
document_ids = vector_store.add_documents(chunks)

print("Documents added:", len(document_ids))
print("Total vectors:", vector_store.index.ntotal)

Documents added: 343
Total vectors: 343


In [76]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [77]:
retriever.invoke("What is the architecture of Llama 2?")

[Document(id='5630d6bf-5602-41bf-a652-b9305f2b5781', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 3, 'page_label': '4'}, page_content='1. Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data. We also\nincreased the size of the pretraining corpus by 40%, doubled the context length of the model, and\nadopted grouped-query attention (Ainslie et al., 2023). We are releasing variants ofLlama 2 with\n7B, 13B, and 70B parameters. We have also trained 34B variants, which we report on in this paper\nbut are not releasing.§\n2. Llama 2-Chat, a fine-tuned version ofLlama 2 that is optimized for d

### Create Local RAG Prompt

In [78]:
prompt = ChatPromptTemplate.from_template(
    """
    You are a question-answering assistant.

    Answer the question only from the supplied context.
    If the answer is not available in the context, say:
    "I do not have enough information in the provided document."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
)

In [79]:
def format_docs(docs):
    return "\n\n".join(
        f"Source: {doc.metadata}\n{doc.page_content}"
        for doc in docs
    )

### Creating RAG Chain

In [83]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | openai_model
    | StrOutputParser()
)

In [84]:
answer = rag_chain.invoke(
    "What is the architecture of Llama 2?"
)

print(answer)

Llama 2 is an auto-regressive language model that uses an optimized transformer architecture.


## Saving Vector Store

In [85]:
vector_store.save_local("faiss_index_llama2")